# Une méthode de calcul du polynôme minimal d'une matrice carrée

## Bases théorique 

Soit une matrice carrée $A\in\mathcal M_d(\mathbb K)$. Pour tout vecteur colonne $X\in\mathcal M_{d,1}(\mathbb K)$, l'ensemble 
$$\langle A,X\rangle =\mathop{Vect}\{A^kX\mid k\in\mathbb N\}$$
est un sev de $E=\mathcal M_{d,1}(\mathbb K)$ et si on note $r$ sa dimension alors $(X,AX,\ldots,A^{r-1}X)$ est une base de $\langle A,X\rangle$. Par ailleurs l'ensemble 
$$I_{A,X}=\{P\in\mathbb K[X]\mid P(A)X=0\}$$
est un idéal non nul de $\mathbb K[X]$. Il existe donc un unique polynôme unitaire $\pi_{A,X}$ tel que 
$$I_{A,X}=\pi_{A,X}\mathbb K[X]$$
On justifie aisément que $r=\deg \pi_{A,X}$. Maintenant on peut démontrer qu'il existe au moins un vecteur $X\in\mathcal M_d(\mathbb K)$ tel que $\pi_{A,X}=\pi_A$. La question qui se pose est avec quelle fréquence le vecteur $X$ vérifiera cette condition si on choisit $X$ au hasard dans un échantillon de $N$ vecteurs. On se propose dans cette feuille d'expérimenter le phénomène est formuler empiriquement une conjoncture. 

Mais d'abords, comment calculer algorithmiquement le polynôme $\pi_{A,X}$. Une méthode simple à mettre en place est disponible : on procède à une élimination de  Gauss-Jordan sur la matrice $M(X)=(X,AX,\ldots,A^{d}X)$ de taille $(d,d+1)$ dont les vecteurs colonnes sont $X,AX,\ldots,A^{d}X$ pour se ramener à une matrice de la forme 
$$MJ(X)=\left(\begin{array}{ccccccc}1 & \cdots & 0 & a_{0} & * & \cdots & * \\ \vdots & \ddots & \vdots & \vdots & \vdots & & \vdots \\ 0 & \cdots & 1 & a_{r-1} & * & \cdots & * \\ 0 & \cdots & 0 & 0 & 0 & \cdots & 0 \\ \vdots & & & \vdots & \vdots & & \vdots \\ 0 & \cdots & 0 & 0 & 0 & \cdots & 0\end{array}\right) $$
On peut alors prouver que le polynôme $\pi_{A,X}$ est donné par 
$$\pi_{A,X}=X^r-a_{r-1}X^{r-1}-\cdots-a_1X-a_0$$
et que $r=\deg\pi_{A,X}=\mathop{rg}(MJ(X))=\mathop{rg}(M(X))$. 

## Test sur un exemple

In [13]:
from sympy import *

On génère aléatoirement une matrice carrée et un vecteur de taille 5

In [77]:
A=randMatrix(5,5,-5,5) ; display(A)

Matrix([
[-1,  2,  3, -3,  5],
[-4, -2, -5, -4, -1],
[-4, -4,  3,  3, -3],
[-1,  3,  0,  3,  5],
[-2, -5,  4, -2,  3]])

In [15]:
# un vecteur alèatoire 
X=randMatrix(5,1,-10,10) ; display(X)

Matrix([
[ 1],
[-8],
[ 2],
[ 2],
[-5]])

On forme maintenant la matrice $M$ dont les colonnes sont $X,AX,A^2X,A^3X,A^4X,A^5X$

In [16]:
# on forme la matrice M(X)
M=X 
Y=X
for i in range(5) :
    Y=A*Y
    M=Matrix.hstack(M,Y)
display(M)

Matrix([
[ 1,  26,  -29,   467, -8936,  10853],
[-8,  11, -203,  1722, -2761,  74121],
[ 2, -12, -390, -1084,  -984, 102390],
[ 2,  13,   81,  -642, -3089, -32211],
[-5, -48, -127,   949, 13170,  57111]])

On calcule le résultat d'une élimination Gauss-Jordan effectuée sur $M$

In [17]:
# élimination de Gauss-Jordan sur M
MJ=M.rref()[0] ; display(MJ)

Matrix([
[1, 0, 0, 0, 0, -3298],
[0, 1, 0, 0, 0, -1834],
[0, 0, 1, 0, 0,  -219],
[0, 0, 0, 1, 0,     4],
[0, 0, 0, 0, 1,    -6]])

On forme maintenant le polynôme minimal $P$ de $A$ en $X$ à l'aides des coefficients sur la colonne $6$ de $MJ$. 

In [18]:
# On peut maintenant former le polynôme minmal en X
x=symbols('x')
P=x**5
for i in range(5) :
    P=P-MJ[i,5]*x**i 
P

x**5 + 6*x**4 - 4*x**3 + 219*x**2 + 1834*x + 3298

On constate qu'à ce stade on obtient un polynôme de degré $5$. Puisque $\pi_{X,A}\mid \pi_A$ et $\deg \pi_A\leqslant 5$ alors cela signifie qu'en fait $\pi_{X,A}=\pi_A=P$. Dans la suite on vérifie que $P$ est bien le polynôme minimal de $A$.

In [19]:
# pour évaluer P en A, on le tronque de son coefficient constant 
P1=P-P.subs(x,0) 

In [20]:
# on vérifie maintenant qu'on  bien P(A)=0
P1.subs(x,A)+P.subs(x,0)*eye(5)

Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

## Fonctions Python pour automatiser la procédure 

Afin d'automatiser toute la démarche on définit dans la suite les fonctions Python `min_X` et `polmin_X` et la méthode `eval_matrix` attachée à l'objet `Poly` :  
- la fonction `min_X` prend en argument la matrice $A$, choisit au hasard un vecteur $X$ et retourne un tuple $(r,M,MJ)$ où $M$ et $MJ$ sont les matrices $M(X)$ et $MJ(X)$ et  $r$ leur rang. 
- la fonction `polmin_X` prend en argument la matrice $A$ et se contente de retourner le polynôme $P$ après l'exécution d'une seule instance de la fonction `min_X`. 
- la méthode `Poly.eval_matrix` permet d'évaluer un polynôme en une matrice (avec la syntaxe `P.eval_matrix(M)`) 

In [97]:
x=symbols('x')
def min_X (A) :
    X=randMatrix(A.rows,1,-10,10) 
    M=X
    for i in range(A.rows) :
        X=A*X
        M=A.hstack(M,X)
    MJ=M.rref()[0]
    return (M.rank(),M,MJ)
def polmin_X (A) :
    r,M,MJ=min_X(A)
    P=x**r
    for i in range(r) :
        P=P-MJ[i,r]*x**i 
    return P.as_poly(x)
def poly_eval_matrix(self, M): # self sera remplacé par l'instance de l'objet qui fait appel à la méthode. 
    if not M.is_square: # tester si la matrice est carrée
        raise NonSquareMatrixError

    if (len(self.gens) != 1): # teste si le polynôme a été déclaré avec une seule indéterminée 
        raise ValueError("un polynôme a une seule indéterminée est attendu")
    expr = self.as_expr() # convertit le polynôme en une expression
    const, non_const = expr.as_independent(self.gens[0]) # récupère la partie constante et la partie variable du polynôme 

    return non_const.subs(self.gens[0], M).doit() + const * eye(M.rows)

Poly.eval_matrix = poly_eval_matrix # c'est tout bête d'ajouter une méthode à un objet existant 


Test préliminaire pour vérifier que `min_X` génère bien des matrices aléatoires de type $M(X)$. 

In [87]:
# on verifie que MinX génère bien des matrices M(X) de façon aléatoire  
for i in range(5) :
   display(min_X(A)[1])

Matrix([
[-10, -22, -227, -131, -3455,   3915],
[ -4, -19,   68, -478,  1996, -10267],
[ -7, -27,    3, -397,   286,  -7315],
[ 10, -22,   90, -626,  2265, -13858],
[  7,  44,   79,  400,  -136,   5682]])

Matrix([
[  8,  -5,  198, -367,  3719, -12083],
[ -5,  37, -123,  696, -3322,  16385],
[ -4,  11,  -45,  449, -1284,   9072],
[-10,  35, -165,  905, -4030,  21030],
[-10, -37,   11, -341,   926,  -6962]])

Matrix([
[-1,  4, -31,   84, -680,  2440],
[-3, -5,  29, -138,  624, -3155],
[ 1, -8,  10,  -66,  277, -1707],
[ 3, -6,  34, -168,  787, -4019],
[ 0,  1,   1,   79, -184,  1276]])

Matrix([
[  6, -46,   78, -981,  2795, -20042],
[-10,  32, -187,  790, -4077,  19705],
[ -8,  15,  -91,  314, -2351,   8542],
[ -4,  47, -217,  996, -5241,  24288],
[  9,  16,  128, -183,  1788,  -6281]])

Matrix([
[-8, -16,  -2, -63,  242, -1342],
[ 2,  -4,   1, 105, -264,  1421],
[ 8,  -1, -46, -18, -139,   741],
[-7,  -6, -31,  57, -388,  1796],
[ 6, -16, -46, -61,  133,  -496]])

On lance maintenant une boucle qui exécute un certain nombre d'instance `min_X(A)` et comptabilise les cas où $\pi_{A,X}=\pi_A$. 

In [88]:
k=0
for i in range(1000) :
    if min_X(A)[0] == 5 :
        k+=1
k


1000

Très difficile d'obtenir quelque chose d'autre que $0$ pour le nombre de cas où $\pi_{A,X}\ne\pi_A$.

In [93]:
P=polmin_X(A) ; P

Poly(x**5 - 16*x**3 + 29*x**2 - 41*x + 33, x, domain='ZZ')

In [98]:
P.eval_matrix(A)

Matrix([
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

## Calcul du polynôme minimal

la fonction `polmin` définie dans la suite est recursive. Elle prend en argument la matrice $A$, lance une instance `polmin_X(A)` pour calculer le polynôme minimal $P$ en un vecteur $X$ choisi aléatoirement et teste ensuite si ce polynôme annule $A$. Si le test échoue elle se relance elle même après incrémentation du compteur `k`. Si le test réussi elle retourne le tuple `(k,P)`

In [116]:
def polmin(A) :
    k=1
    P=polmin_X(A)
    #display(P.eval_matrix(A) == zeros(A.rows,A.cols))
    if P.eval_matrix(A) == zeros(A.rows,A.cols) : 
        return (k,P)
    else :
        k+=1
        polmin(A)

In [117]:
polmin(A)

(1, Poly(x**5 - x**4 - 6*x**3 + 7*x**2 - 72*x, x, domain='ZZ'))

## Fréquence d'apparition d'une matrice cyclique ($\pi_A=\chi_A$)

Dans la suite on lance une boucle qui génère des matrices aléatoires $A$ et affiche le résultat de `polmin(A)`. Le but est cette fois de « mesurer » la fréquence avec laquelle on obtient une matrice $A$ telle que $\pi_A=\chi_A$ (une matrice cyclique) en choisissant au hasard la matrice $A$.

In [118]:
compt_noncyclique=0
L=[]
for _ in range (100) :
    A=randMatrix(5,5,-100,100)
    r,P=polmin(A)
    L.append(r)
    if P.degree() < 5 :
       compt_noncyclique += 1
print(compt_noncyclique,L)

0 [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


100% de matrice cycliques sur 100 matrices quand les coefficients sont pris aléatoirement dans l'intervalle des entiers $\lbrack -100,100\rbrack$

In [123]:
compt_noncyclique=0
L=[]
for _ in range (100) :
    A=randMatrix(5,5,-2,2)
    display(polmin(A))
    r,P=polmin(A)
    L.append(r)
    if P.degree() < 5 :
        compt_noncyclique += 1
print(compt_noncyclique)

(1, Poly(x**5 + 3*x**4 - 8*x**3 - 40*x**2 - 55*x - 18, x, domain='ZZ'))

(1, Poly(x**5 + 5*x**4 + 11*x**3 + 8*x**2 - 25*x - 2, x, domain='ZZ'))

(1, Poly(x**5 - x**4 - 10*x**3 + 11*x**2 - 7*x + 22, x, domain='ZZ'))

(1, Poly(x**5 + 7*x**4 + 14*x**3 + 26*x**2 + 4*x - 88, x, domain='ZZ'))

(1, Poly(x**5 + x**4 + 3*x**3 - 14*x**2 - 10*x + 21, x, domain='ZZ'))

...

(1, Poly(x**5 - x**4 - 7*x**3 - 59*x**2 + 6*x + 24, x, domain='ZZ'))

(1, Poly(x**5 + 4*x**4 + 19*x**3 + 44*x**2 + 55*x + 12, x, domain='ZZ'))

(1, Poly(x**5 - 5*x**4 + x**3 - 2*x**2 - 70*x + 48, x, domain='ZZ'))

(1, Poly(x**5 + 3*x**4 - 2*x**3 - 12*x**2 - 8*x, x, domain='ZZ'))

TypeError: cannot unpack non-iterable NoneType object

À peine 1, rarement 2 matrices sur 100 quand les coefficients sont aléatoirement pris dans $\{-2,-1,0,1,2\}$

In [125]:
r,P=polmin(A)